# 03 · ReAct, plan-and-execute and reflection

Implement the three reasoning patterns interviewers ask about: text **ReAct** (Thought / Action / Observation) with a parser and the stop sequence that keeps it honest, the same agent with native tool calling, **plan-and-execute** (a structured-output planner, an executor that runs a dependency graph, a re-planner on failure), and a **reflection** loop that revises against a verifier. Then decide which to use when.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · The agent loop from scratch](01_agent_loop_from_scratch.ipynb)

## Why this matters in interviews

- "What is ReAct?" is asked in almost every agent interview; the follow-up is "what stops the model inventing the observation?"
- Plan-and-execute vs ReAct is a cost and control question: how many expensive calls, where a human can approve, what can run in parallel.
- Reflection sounds free and is not: it multiplies calls, and it only helps when something can *check* the output.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag08` | What is ReAct? |
| `pr21` | What is the ReAct prompt format and why does it work? |
| `ag09` | What is plan-and-execute, and when is it better than ReAct? |
| `ag33` | What is the ReWOO pattern and why does it save money? |
| `ag10` | What is reflection, and what does it cost? |
| `ag11` | What are the main agentic design patterns? |
| `ag44` | How do you decide when a task is "done"? |
| `pr31` | What is a scratchpad, and why does it help an agent? |

## 1. Three shapes of reasoning

**In plain English:**
- **ReAct** thinks one step at a time: think, act, look at the result, think again.
- **Plan-and-execute** writes the whole plan first, then carries it out, and only re-plans if a step fails.
- **Reflection** writes a draft, checks it, and rewrites it.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

import re
import json
import ast
import operator
import difflib
import random
import datetime as dt
from typing import Literal

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pydantic import BaseModel, Field

random.seed(0)
np.random.seed(0)


def draw_patterns():
    rows = [("ReAct", ["Thought", "Action", "Observation", "...repeat...", "Final Answer"]),
            ("Plan-and-execute", ["Plan\n(1 LLM call)", "run steps\n(no LLM needed)", "re-plan\nonly on failure",
                                  "Solve\n(1 LLM call)"]),
            ("Reflection", ["Draft", "Verify /\ncritique", "Revise\n(cap: 2)", "Final"])]
    fig, ax = plt.subplots(figsize=(11, 3.6))
    colors = ["#e7f0ff", "#ebfbee", "#fff4e6"]
    for r, (name, steps) in enumerate(rows):
        y = (len(rows) - 1 - r) * 1.25
        ax.text(-0.25, y + 0.4, name, ha="right", va="center", fontsize=10, weight="bold")
        for i, s in enumerate(steps):
            x = i * 2.1
            ax.add_patch(FancyBboxPatch((x, y), 1.75, 0.8, boxstyle="round,pad=0.05", fc=colors[r], ec="#495057"))
            ax.text(x + 0.87, y + 0.4, s, ha="center", va="center", fontsize=8)
            if i < len(steps) - 1:
                ax.annotate("", xy=(x + 2.08, y + 0.4), xytext=(x + 1.8, y + 0.4),
                            arrowprops=dict(arrowstyle="->", lw=1.4))
    ax.annotate("", xy=(0.9, 2.55 + 0.8), xytext=(5.1, 2.55 + 0.8),
                arrowprops=dict(arrowstyle="->", lw=1.2, color="#c92a2a", connectionstyle="arc3,rad=0.25"))
    ax.set_xlim(-3.3, 10.4); ax.set_ylim(-0.2, 3.8); ax.axis("off")
    ax.set_title("ReAct decides after every observation; plan-and-execute decides up front; reflection adds a review pass")
    plt.show()


draw_patterns()

## 2. The tools (same as notebook 01)

A frozen clock (12:00 UTC on 2026-09-29), a safe calculator and a four-passage knowledge base. Each tool takes one string, which keeps the text-ReAct parser simple.

In [ ]:
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.USub: operator.neg}


def calculator(expression: str) -> str:
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
            return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only arithmetic is allowed")
    v = ev(ast.parse(expression, mode="eval").body)
    return str(int(v) if float(v).is_integer() else round(v, 4))


UTC_OFFSET = {"London": 1, "Tokyo": 9, "New York": -4}
NOW = dt.datetime(2026, 9, 29, 12, 0, tzinfo=dt.timezone.utc)


def get_time(city: str) -> str:
    city = city.strip()
    if city not in UTC_OFFSET:
        raise KeyError(f"unknown city {city!r}; known cities: {', '.join(UTC_OFFSET)}")
    return f"{city}: {NOW + dt.timedelta(hours=UTC_OFFSET[city]):%H:%M} local time"


KB = {"refunds": "Refunds are issued within 14 days of purchase for unused items.",
      "shipping": "Standard shipping takes 3-5 business days; express shipping takes 1 business day.",
      "warranty": "The warranty covers manufacturing defects for 2 years.",
      "returns": "Items can be returned within 30 days in their original packaging."}
_STOP = set("a an the is are do does how what when for of to in on my i it and or can many have".split())


def search_kb(query: str) -> str:
    q = {w.strip("?.,!").lower() for w in query.split()} - _STOP
    scored = sorted(((len(q & ({w.strip("?.,;!").lower() for w in text.split()} | {d, d.rstrip("s")})), d, text)
                     for d, text in KB.items()), reverse=True)
    hits = [f"[{d}] {t}" for s, d, t in scored if s > 0][:2]
    return "\n".join(hits) if hits else f"No passages match {query!r}."


TOOLS_BY_NAME = {"get_time": get_time, "calculator": calculator, "search_kb": search_kb}


def run_tool(name: str, tool_input: str) -> str:
    if name not in TOOLS_BY_NAME:
        return f"ERROR: unknown tool {name!r}; use one of {list(TOOLS_BY_NAME)}"
    try:
        return TOOLS_BY_NAME[name](tool_input)
    except Exception as e:
        return f"ERROR {type(e).__name__}: {e}"


print(run_tool("get_time", "Tokyo"), "|", run_tool("calculator", "24 - 21"), "|", run_tool("get_time", "Tokio"))

## 3. Text ReAct: a prompt format plus a parser

**ReAct** (Reason + Act, Yao et al., 2022) asks the model to write in a fixed pattern. Your code reads the `Action` lines, runs the tool, and appends the real `Observation`. The model's `Thought` lines are its scratchpad: the reasoning is visible and the actions are parseable.

In [ ]:
REACT_SYSTEM = """Answer the question by interleaving Thought, Action and Observation steps.
Tools:
- get_time: current local time in a city. Input: a city name.
- calculator: arithmetic. Input: an expression like 24 - 21.
- search_kb: company policies. Input: a search query.
Use exactly this format:
Thought: your reasoning
Action: the tool name
Action Input: the input
Observation: the tool result (written by the system, never by you)
... repeat Thought/Action/Action Input/Observation as needed ...
Thought: I now know the answer
Final Answer: the answer"""

ACTION_RE = re.compile(r"Action:\s*(\w+)\s*\nAction Input:\s*(.+?)\s*$", re.S)
FINAL_RE = re.compile(r"Final Answer:\s*(.+)", re.S)


def parse_react(text: str):
    """Returns ('final', answer) | ('action', tool, input) | ('error', why). Final Answer wins if present."""
    if m := FINAL_RE.search(text):
        return ("final", m.group(1).strip())
    if m := ACTION_RE.search(text):
        return ("action", m.group(1), m.group(2).strip().splitlines()[0])
    return ("error", "Invalid format. Write 'Action:' and 'Action Input:' lines, or 'Final Answer:'.")


def apply_stop(text: str, stop=("Observation:",)) -> str:
    """What the API's `stop` parameter does: generation ends right before the stop string."""
    cuts = [text.index(s) for s in stop if s in text]
    return text[:min(cuts)] if cuts else text


print(parse_react("Thought: need the time\nAction: get_time\nAction Input: Tokyo"))
print(parse_react("Thought: done\nFinal Answer: 3 hours"))
print(parse_react("I think it is about 3 hours."))

To see the format offline we need something that *writes* ReAct text. `scripted_policy` plays the model's part deterministically: it reads the transcript so far and writes the next step, using the observations it was given. Its first reply also does what an unconstrained model often does: it keeps generating past the action and **invents an observation and an answer**.

In [ ]:
def scripted_policy(transcript: str, stop=("Observation:",)) -> str:
    obs = re.findall(r"Observation: (.*)", transcript)
    if not obs:
        text = ("Thought: I need the current time in Tokyo first.\nAction: get_time\nAction Input: Tokyo\n"
                "Observation: Tokyo: 18:00 local time\nThought: So 6 hours are left.\nFinal Answer: 6 hours")
    elif len(obs) == 1:
        hour = int(re.search(r"(\d{1,2}):\d{2}", obs[0]).group(1))
        text = f"Thought: It is {hour}:00 there and midnight is 24:00.\nAction: calculator\nAction Input: 24 - {hour}"
    else:
        text = f"Thought: I now know the answer.\nFinal Answer: It is {obs[0].split(': ', 1)[1]} in Tokyo, so {obs[1]} hours until midnight."
    return apply_stop(text, stop)


def react_loop(question, llm, stop=("Observation:",), max_steps=5, show=True):
    transcript = f"Question: {question}\n"
    for step in range(1, max_steps + 1):
        out = llm(transcript, stop=stop).strip()
        transcript += out + "\n"
        parsed = parse_react(out)
        if parsed[0] == "final":
            status, answer = "completed", parsed[1]
            break
        obs = run_tool(parsed[1], parsed[2]) if parsed[0] == "action" else parsed[1]
        transcript += f"Observation: {obs}\n"
    else:
        status, answer = "max_steps", None
    if show:
        print(transcript)
    return {"status": status, "answer": answer, "steps": step, "transcript": transcript}


QUESTION = "What time is it in Tokyo, and how many hours until midnight there?"
honest = react_loop(QUESTION, scripted_policy)
assert honest["answer"].endswith("3 hours until midnight.") and honest["steps"] == 3

Three model turns: look up the time, compute `24 - 21`, answer. Every `Observation:` line was written by *our code* from a real tool call. The first turn's invented "Observation: Tokyo: 18:00" never reached the transcript, because the stop sequence cut generation just before it.

Now the same run without a stop sequence:

In [ ]:
no_stop = react_loop(QUESTION, scripted_policy, stop=(), max_steps=5, show=False)
print("status:", no_stop["status"], "| answer:", no_stop["answer"], "| steps:", no_stop["steps"])
assert no_stop["answer"] == "6 hours", "without the stop sequence the invented observation becomes the answer"

The run "finished" in one step with **a confident answer built on an observation that never happened**. The real tool was never called. This is the single most common bug in a hand-rolled ReAct loop (`ag08`): if you see plausible tool results that never ran, check the stop sequence first. With native tool calling the API enforces this structurally: the model *cannot* write the `tool` message.

### A model that does not follow the format

Now the real client with the same prompt and `stop=["Observation:"]`:

In [ ]:
def chat_llm(transcript, stop=("Observation:",)):
    resp = client.chat.completions.create(model=MODEL, stop=list(stop) or None, messages=[
        {"role": "system", "content": REACT_SYSTEM}, {"role": "user", "content": transcript}])
    return resp.choices[0].message.content or ""


live = react_loop(QUESTION, chat_llm, max_steps=3)
print("status:", live["status"])

Offline, the stand-in does not speak the ReAct protocol, so every turn is a format error, and the loop feeds that back ("Invalid format ...") until `max_steps`. A real model such as gpt-4.1-mini follows the format most of the time, and "most of the time" is the problem: format drift, a missing `Action Input`, or a JSON-ish input the regex does not expect. That is why production agents moved to **native tool calling**, where the call is structured data, not text to parse.

## 4. The same agent with native tool calling

Same tools, now described as JSON schemas; the loop from notebook 01 in compact form.

In [ ]:
def schema(name, description, param):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {param: {"type": "string"}}, "required": [param], "additionalProperties": False}}}


NATIVE_TOOLS = [schema("get_time", "Get the current local time in a city.", "city"),
                schema("calculator", "Evaluate an arithmetic expression such as 24 - 21.", "expression"),
                schema("search_kb", "Search company policies: refunds, shipping, warranty, returns.", "query")]


def native_agent(question, max_steps=6):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=NATIVE_TOOLS).choices[0].message
        if not msg.tool_calls:
            return msg.content, step, messages
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [t.model_dump() for t in msg.tool_calls]})
        for t in msg.tool_calls:
            arg = next(iter(json.loads(t.function.arguments).values()))
            messages.append({"role": "tool", "tool_call_id": t.id, "content": run_tool(t.function.name, arg)})
    return None, max_steps, messages


answer, steps, msgs = native_agent(QUESTION)
print(f"{steps} LLM calls ->", answer)

Offline the stand-in makes one round of tool calls (only `get_time` matches the words of the question) and quotes the result; a real model would also compute 24 - 21 (with the calculator or in its head). The loop is the same as ReAct's; only the protocol changed.

| | Text ReAct | Native tool calling |
|---|---|---|
| action format | text you parse with regex | structured `tool_calls` (JSON) |
| stop the model inventing results | your `stop=["Observation:"]` | enforced by the API (`tool` role) |
| visible reasoning | the `Thought:` lines | optional `content` text, or hidden reasoning tokens in reasoning models |
| parallel actions | awkward | several `tool_calls` in one message |
| works with any text model | yes | needs a model trained for tool use |
| use today | teaching, very small or legacy models | production |

The **scratchpad** (`pr31`) is the same in both: the running transcript of thoughts, actions and observations, carried into every next step. Keep it from exploding by shaping observations and dropping superseded failed attempts (`ag37`).

## 5. Plan-and-execute

**In plain English:** ask a strong model once for the whole plan, then run the steps with cheap code (or a cheap model), and only go back to the planner when a step fails.

The plan is **structured output**: a pydantic model the planner must fill. Steps can depend on earlier steps and reference their results as `{step_N}`, which is the **ReWOO** idea (`ag33`): plan the tool calls *with placeholders* before any result exists, so no LLM call is needed between tool steps.

In [ ]:
class Step(BaseModel):
    id: int
    tool: Literal["get_time", "search_kb", "calculator"]
    input: str = Field(description="the tool input; may use an earlier result as {step_N}")
    depends_on: list[int] = Field(default_factory=list)


class Plan(BaseModel):
    goal: str
    steps: list[Step]


PLANNER_SYSTEM = ("Break the task into tool steps. Tools: get_time(city), search_kb(query), calculator(expression). "
                  "Use {step_N} to reference the result of step N and list it in depends_on.")


def llm_planner(task, feedback=None) -> Plan:
    user = task if feedback is None else f"{task}\n\nThe previous plan failed at step {feedback['step'].id}: {feedback['error']}"
    resp = client.chat.completions.parse(model=MODEL, response_format=Plan, messages=[
        {"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": user}])
    return resp.choices[0].message.parsed


TASK = ("A customer in Tokio buys 3 items at 19.99 EUR each. What time is it there, how long does express "
        "shipping take, and what is the total including 20% VAT?")
llm_plan = llm_planner(TASK)
print(llm_plan.model_dump_json(indent=1))

Structured output guarantees the plan *parses*. It does not guarantee the plan *makes sense*. Offline, the stand-in fills the schema from words and numbers in the prompt, so you get one step that depends on a step "19" that does not exist. A real model gives a sensible plan most of the time, and you still validate every plan before running it:

In [ ]:
PLACEHOLDER = re.compile(r"\{step_(\d+)\}")
MAX_PLAN_STEPS = 8


def validate_plan(plan: Plan) -> list[str]:
    problems, seen = [], set()
    if not 1 <= len(plan.steps) <= MAX_PLAN_STEPS:
        problems.append(f"plan has {len(plan.steps)} steps; allowed 1-{MAX_PLAN_STEPS}")
    for s in plan.steps:
        if s.id in seen:
            problems.append(f"duplicate step id {s.id}")
        for d in s.depends_on:
            if d not in seen:
                problems.append(f"step {s.id} depends on step {d}, which does not come before it")
        for ref in map(int, PLACEHOLDER.findall(s.input)):
            if ref not in s.depends_on:
                problems.append(f"step {s.id} uses {{step_{ref}}} without listing it in depends_on")
        seen.add(s.id)
    return problems


print("LLM plan problems:", validate_plan(llm_plan) or "none")

To demonstrate the executor and the re-planner deterministically, `stub_planner` writes the plan a capable model should write for this task. It copies the customer's typo "Tokio", as a real planner would, because the typo is in the task. When it receives failure feedback it fixes the city from the error message, which is what we want an LLM re-planner to do. Swap `PLANNER = llm_planner` once a real model is connected; the executor does not care who wrote the plan.

In [ ]:
def stub_planner(task, feedback=None) -> Plan:
    city = "Tokio"
    if feedback and "known cities" in feedback["error"]:
        known = feedback["error"].split("known cities: ")[1].split(", ")
        city = difflib.get_close_matches(city, known, n=1)[0]
    return Plan(goal="answer the customer's three questions", steps=[
        Step(id=1, tool="get_time", input=city),
        Step(id=2, tool="search_kb", input="express shipping time"),
        Step(id=3, tool="calculator", input="3 * 19.99"),
        Step(id=4, tool="calculator", input="{step_3} * 1.2", depends_on=[3])])


def waves(plan: Plan) -> list[list[int]]:
    """Group steps into levels: every step in a level only depends on earlier levels -> they can run in parallel."""
    level = {}
    for s in plan.steps:
        level[s.id] = 1 + max((level[d] for d in s.depends_on), default=-1)
    return [[sid for sid, lv in level.items() if lv == k] for k in range(max(level.values()) + 1)]


plan = stub_planner(TASK)
assert validate_plan(plan) == []
print("waves:", waves(plan), "-> steps in the same wave are independent and could run in parallel")

In [ ]:
SOLVER_SYSTEM = "Write the final answer to the task using only the evidence. Be brief."


def execute_plan(task, planner, max_replans=1):
    plan, results, replans, llm_calls = planner(task), {}, 0, 1
    while True:
        if problems := validate_plan(plan):
            return {"status": "invalid_plan", "problems": problems, "llm_calls": llm_calls}
        failure = None
        for wave in waves(plan):
            for sid in wave:
                step = next(s for s in plan.steps if s.id == sid)
                if sid in results:
                    continue                                   # done before the re-plan
                tool_input = step.input.format(**{f"step_{k}": v for k, v in results.items()})
                out = run_tool(step.tool, tool_input)
                print(f"  step {sid}: {step.tool}({tool_input!r}) -> {out[:70]}")
                if out.startswith("ERROR"):
                    failure = {"step": step, "error": out}
                    break
                results[sid] = out
            if failure:
                break
        if failure is None:
            break
        if replans == max_replans:
            return {"status": "failed", "error": failure["error"], "results": results, "llm_calls": llm_calls}
        replans += 1
        llm_calls += 1
        print(f"  -> re-planning after step {failure['step'].id} failed")
        plan = planner(task, feedback=failure)
    evidence = "\n".join(f"step {sid}: {results[sid]}" for sid in sorted(results))
    resp = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": SOLVER_SYSTEM}, {"role": "user", "content": f"Task: {task}\n\nEvidence:\n{evidence}"}])
    llm_calls += 1
    return {"status": "completed", "answer": resp.choices[0].message.content, "results": results,
            "replans": replans, "llm_calls": llm_calls}


PLANNER = stub_planner
run = execute_plan(TASK, PLANNER)
print("\nstatus:", run["status"], "| replans:", run["replans"], "| LLM calls:", run["llm_calls"])
print("results:", run["results"])
print("answer :", run["answer"])
assert run["status"] == "completed" and run["results"][1] == "Tokyo: 21:00 local time"
assert run["results"][4] == calculator("59.97 * 1.2")

Read the trace:

- Wave 1 started with step 1, `get_time('Tokio')`, which failed with an instructive error naming the known cities.
- The executor called the planner again with that feedback; the new plan fixed the city. Step 1 succeeded and the remaining steps ran; step 4 used step 3's result through the `{step_3}` placeholder.
- The solver wrote the answer from the evidence. Offline it quotes the best-matching evidence line; a real model combines all four facts into one sentence.

Count the model calls: **one plan + one re-plan + one solve = 3**, for five tool executions. A ReAct agent needs a model call before *every* tool call plus one to answer. That difference is the ReWOO saving; it grows with the number of tool steps, and the calls that remain are the ones worth sending to the strong model. Handing the invalid LLM plan to the same executor stops before any tool runs:

In [ ]:
bad = execute_plan(TASK, llm_planner)
print(bad["status"], bad.get("problems"))
assert bad["status"] in ("invalid_plan", "completed")

react_calls = len(run["results"]) + 1 + 1                    # one call per tool step (+1 for the failed try) + final
print(f"model calls - ReAct (estimated): {react_calls} | plan-and-execute: {run['llm_calls']}")

n_steps = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(n_steps, n_steps + 1, "o-", label="ReAct: a model call before every tool call, +1 to answer")
ax.plot(n_steps, np.full_like(n_steps, 2), "s-", label="plan-and-execute / ReWOO: plan + solve")
ax.plot(n_steps, np.full_like(n_steps, 3), "s--", alpha=0.6, label="... plus one re-plan")
ax.scatter([4], [run["llm_calls"]], s=120, facecolors="none", edgecolors="#c92a2a", zorder=5, label="the run above")
ax.set_xlabel("tool steps in the task"); ax.set_ylabel("model calls")
ax.set_title("Planning up front keeps model calls flat as the task grows")
ax.legend(fontsize=8)
plt.show()

When is plan-and-execute better (`ag09`)? When the task has a **known shape** (gather these facts, then answer), when there are **side effects** (a human approves the plan before anything happens), when steps can **run in parallel**, and when **cost must be predictable**. Its weakness: a plan written before seeing any data can be wrong from step one, so it needs an explicit re-plan trigger and a cap on re-plans. The usual winner in practice is a hybrid: coarse plan, each step run by a small ReAct loop, re-plan only when a step reports failure.

## 6. Reflection: draft, verify, revise

**In plain English:** check the answer and fix it before showing it. It helps most when the check is **mechanical** (tests pass, JSON parses, the citation exists), because then the feedback is concrete. Self-critique with no ground truth mostly produces more confident prose (`ag10`).

The verifier below checks an answer to a policy question: it must cite a source like `[returns]`, the source must exist, and every number in the answer must appear in the cited passage.

In [ ]:
def verify(answer: str) -> list[str]:
    problems = []
    tags = re.findall(r"\[(\w+)\]", answer or "")
    if not tags:
        problems.append("no citation: cite the policy source, e.g. [returns]")
    for t in tags:
        if t not in KB:
            problems.append(f"cited source [{t}] does not exist")
    cited = " ".join(KB.get(t, "") for t in tags)
    for num in re.findall(r"\b\d+\b", answer or ""):
        if tags and num not in cited:
            problems.append(f"the number {num} is not in the cited passage")
    return problems


def reflect(question, max_revisions=2):
    calls = 1
    answer = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}]).choices[0].message.content
    rounds = [(answer, verify(answer))]
    while rounds[-1][1] and len(rounds) <= max_revisions:
        problems = "\n".join(f"- {p}" for p in rounds[-1][1])
        prompt = (f"Your previous answer failed review:\n{problems}\nRewrite it using ONLY these passages and cite "
                  f"the source tag.\n\nPassages:\n{search_kb(question)}\n\nQuestion: {question}")
        answer = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}]).choices[0].message.content
        calls += 1
        rounds.append((answer, verify(answer)))
    return rounds, calls


rounds, calls = reflect("How many days do I have to return an item?")
for i, (ans, problems) in enumerate(rounds):
    print(f"round {i}: {ans[:90]}\n         verifier: {problems or 'PASS'}")
print("LLM calls:", calls)
assert rounds[-1][1] == [] and calls <= 3

The draft had no citation (offline it is the stand-in's canned reply; a real model might confidently say "30 days" without a source, which is worse because it *looks* right). The verifier's concrete complaint plus retrieved passages produced a grounded, cited answer on the next round. Two things to say in an interview:

- **Reflection against a verifier beats reflection against the model's own judgement.** Run the tests, the validator, the citation check, and feed the *specific* failure back. (Tool-backed critique is the idea behind CRITIC, Gou et al. 2023.)
- **It costs calls and latency**: here 2 calls instead of 1, and with an LLM critic 3 or more. Cap the revisions (2 is usually enough) and measure whether round 2 actually improves your metric.

**Reflexion** (Shinn et al., 2023) is the across-attempts version: after a failed episode the agent writes a short lesson ("I used the wrong city name") into memory and reads it on the next attempt.

## 7. Which pattern when?

| | ReAct | Plan-and-execute / ReWOO | Reflection / Reflexion | Workflow (fixed graph) |
|---|---|---|---|---|
| decides the next step | after each observation | up front; re-plans on failure | adds a review pass to any of these | you, at design time |
| expensive model calls | ~1 per step | 1 plan + 1 solve (+ re-plans) | ×2-3 | fixed |
| adaptivity | high | lower | n/a | none |
| cost predictability | poor without caps | good | cap the iterations | exact |
| human approval | awkward, mid-loop | natural: approve the plan | review the final draft | at fixed nodes |
| parallelism | hard | easy (dependency waves) | n/a | designed in |
| best for | exploration, unknown shape | known-shape tasks, side effects, sign-off | outputs a program can check | known procedures |
| typical failure | loops, format drift, invented observations | stale plan made before seeing data | confident prose without ground truth | brittle on new cases |

**When is it done (`ag44`)?** Rank the stop signals by reliability: a programmatic post-condition (the record exists, the verifier passes) > a structured "finish" call you validate > "no tool call in the reply" > budget exhausted (a failure, and must be logged as one).

## Try it yourself

**Exercise 1.** Make `parse_react` tolerant of JSON action inputs such as `Action Input: {"city": "Tokyo"}` (use the first value), and test it.

In [ ]:
# Solution — try it yourself first, then run this
def parse_react_json(text: str):
    kind = parse_react(text)
    if kind[0] == "action" and kind[2].startswith("{"):
        try:
            return ("action", kind[1], str(next(iter(json.loads(kind[2]).values()))))
        except (json.JSONDecodeError, StopIteration):
            return ("error", "Action Input is not valid JSON")
    return kind


print(parse_react_json('Thought: x\nAction: get_time\nAction Input: {"city": "Tokyo"}'))
assert parse_react_json('Action: get_time\nAction Input: {"city": "Tokyo"}') == ("action", "get_time", "Tokyo")

**Exercise 2.** Run each wave's steps concurrently with `concurrent.futures.ThreadPoolExecutor` and check that the results equal the sequential run (use the corrected plan, `stub_planner(TASK, feedback=...)`).

In [ ]:
# Solution — try it yourself first, then run this
from concurrent.futures import ThreadPoolExecutor

fixed = stub_planner(TASK, feedback={"error": "known cities: London, Tokyo, New York"})
parallel_results = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    for wave in waves(fixed):
        steps = [s for s in fixed.steps if s.id in wave]
        inputs = [s.input.format(**{f"step_{k}": v for k, v in parallel_results.items()}) for s in steps]
        for s, out in zip(steps, pool.map(run_tool, [s.tool for s in steps], inputs)):
            parallel_results[s.id] = out
print(parallel_results)
assert parallel_results == run["results"]

**Exercise 3.** What does reflection cost? Run `reflect` on the three questions below and report the pass rate and the average number of model calls per question, compared with 1 call for a plain answer.

In [ ]:
# Solution — try it yourself first, then run this
questions = ["How many days do I have to return an item?", "How long does the warranty last?", "When are refunds issued?"]
outcomes = [reflect(q) for q in questions]
passed = sum(not rounds[-1][1] for rounds, _ in outcomes)
avg_calls = np.mean([calls for _, calls in outcomes])
for q, (rounds, calls) in zip(questions, outcomes):
    print(f"{q:45s} -> {calls} calls, final: {rounds[-1][0][:60]}")
print(f"pass rate {passed}/{len(questions)} | average model calls {avg_calls:.1f} (x{avg_calls:.1f} a single call)")

## Say it in an interview

**ReAct in 30 seconds (`ag08`, `pr21`).** "The model alternates Thought, Action and Action Input; my code runs the tool and appends the real Observation, and the model continues until Final Answer. The critical detail is the stop sequence at `Observation:` so the model cannot invent tool results. Native tool calling replaced the text format in production because structured calls are far more reliable to parse, and the API enforces that only my code writes tool results; the loop is identical."

**Plan-and-execute (`ag09`, `ag33`).** "One strong-model call writes a plan, validated against a schema and against sanity rules; steps run with cheap code or a cheap model; placeholders let later steps use earlier results without another model call (ReWOO); a failed step triggers a bounded re-plan. It wins on known-shape tasks with side effects because a human can approve the plan and independent steps parallelise. It loses on exploration, because the plan was written before seeing any data."

**Reflection (`ag10`).** "Draft, critique, revise, capped at two rounds. It works when a verifier gives concrete feedback: tests, validators, citation checks. Without ground truth it mostly increases confidence. It costs 2-3× the calls, so measure whether it moves the metric."

**Traps.** No stop sequence (invented observations); trusting a schema-valid plan without validating it; unbounded re-plans or revisions; calling "no tool call" a success; listing patterns as a menu instead of starting simple and adding one only for a specific failure (`ag11`).

## Next

- [04 · Agent memory](04_agent_memory_short_and_long_term.ipynb) · [05 · Multi-agent patterns](05_multi_agent_patterns.ipynb) · [06 · Guardrails, HITL and failure modes](06_guardrails_hitl_and_failure_modes.ipynb).
- Frameworks: [LangGraph ReAct agent](../14_langgraph/05_react_tool_calling_agent.ipynb), [conditional edges and loops](../14_langgraph/02_conditional_edges_and_loops.ipynb), [LangChain agents](../13_langchain/06_agents_and_memory.ipynb).
- [Structured outputs with pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb) · [Agent evaluation](../12_evaluation/04_agent_evaluation.ipynb).
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) (chapters *ReAct*, *Planning*, *Reflection*) and the [interview bank](../../ai_interview_prep/index.html).